# 01 – Khảo sát cấu trúc dữ liệu thô (Bước 1)

**Mục tiêu:**
- Khảo sát metadata & cấu trúc dữ liệu thô của **dữ liệu địa từ KAK** (HAPI / IAGA-2002) và **dữ liệu động đất USGS** (CSV / GeoJSON).
- Báo cáo số dòng, số cột, kiểu dữ liệu, thống kê mô tả (min, max, mean, std), khoảng thời gian phủ dữ liệu.
- Phân nhóm cột: Thời gian / Định danh / Đo lường / Chất lượng / Metadata.

**Nguyên tắc:** Chỉ đọc dữ liệu, KHÔNG chỉnh sửa file gốc trong `data/raw/`.


In [ ]:
# ── Setup môi trường & nạp thư viện ──────────────────────────────────────────
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Thêm đường dẫn src vào sys.path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import config as cfg
from parsers import (
    parse_iaga2002_file, parse_iaga2002_header,
    load_all_intermagnet, load_usgs_csv, load_usgs_geojson
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_RAW:     {cfg.DATA_RAW}')


## 1. KAK (IAGA-2002 / HAPI) – Đọc Header & Metadata file thô


In [ ]:
# Đọc header của file dữ liệu thô KAK đầu tiên tìm được
# Mục đích: Đọc thông tin metadata trạm (tọa độ, độ cao, thành phần đo, sentinel) mà không nạp toàn bộ dữ liệu

raw_intermagnet = cfg.DATA_RAW / 'intermagnet'

station_metas = {}
for station in cfg.STATIONS:
    files = sorted(raw_intermagnet.glob(f'intermagnet_{station}_*.min'))
    if not files:
        print(f'⚠  Không tìm thấy file cho trạm {station}')
        continue

    # Đọc header của file đầu tiên
    first_file = files[0]
    print(f'\n{"─"*60}')
    print(f'Trạm {station} | File: {first_file.name}')
    print(f'{"─"*60}')

    raw_text = first_file.read_text(encoding='ascii', errors='replace')
    header_lines = []
    for line in raw_text.splitlines():
        if line.strip().upper().startswith('DATE'):
            header_lines.append(line)
            break
        header_lines.append(line)

    # In header gốc
    print('\n=== HEADER GỐC (30 dòng đầu) ===')
    for i, ln in enumerate(header_lines[:30]):
        print(f'{i+1:3d}: {ln}')

    # Parse metadata bằng hàm parse_iaga2002_header
    meta = parse_iaga2002_header(header_lines)
    station_metas[station] = meta

    print('\n=== METADATA ĐÃ PARSE ===')
    for k, v in meta.items():
        print(f'  {k:30s}: {v}')


## 2. KAK (IAGA-2002 / HAPI) – Nạp dữ liệu và báo cáo thống kê


In [ ]:
# Nạp toàn bộ dữ liệu chuỗi thời gian địa từ cho trạm KAK
# Dữ liệu thô giữ nguyên giá trị Sentinel (ví dụ 99999.00) để phục vụ bước đếm chất lượng (Bước 2)

station_dfs   = {}  # Lưu DataFrame thô
station_meta2 = {}  # Lưu metadata trạm

for station in cfg.STATIONS:
    try:
        df, meta = load_all_intermagnet(station)
        station_dfs[station]   = df
        station_meta2[station] = meta
        print(f'\n✅ {station}: Nạp xong {len(df):,} dòng')
    except FileNotFoundError as e:
        print(f'⚠  {e}')


In [ ]:
# ── Báo cáo cấu trúc DataFrame KAK ───────────────────────────────────────────
for station, df in station_dfs.items():
    meta = station_meta2.get(station, {})
    print(f'\n{"="*65}')
    print(f'  TRẠM {station} – Báo cáo cấu trúc dữ liệu thô')
    print(f'{"="*65}')

    print(f'\n📌 Tổng quan:')
    print(f'   Số dòng  : {len(df):,}')
    print(f'   Số cột   : {len(df.columns)}')
    print(f'   Danh sách: {list(df.columns)}')

    t_min = df["time_utc"].min()
    t_max = df["time_utc"].max()
    print(f'   Từ       : {t_min}')
    print(f'   Đến      : {t_max}')
    print(f'   Thời gian: {t_max - t_min}')

    print(f'\n📌 Kiểu dữ liệu (Data Types):')
    print(df.dtypes.to_string())

    print(f'\n📌 Thống kê mô tả (vẫn còn chứa giá trị Sentinel):')
    print(df.describe(include='all').T.to_string())

    print(f'\n📌 5 dòng dữ liệu đầu tiên:')
    print(df.head().to_string())


In [ ]:
# ── Phân nhóm các cột trong dữ liệu KAK ──────────────────────────────────────
print('\n=== PHÂN NHÓM CỘT DỮ LIỆU KAK ===')
for station, df in station_dfs.items():
    print(f'\nTrạm {station}:')
    col_groups = {
        'Thời gian' : [c for c in df.columns if 'time' in c.lower() or 'doy' in c.lower()],
        'Định danh' : [c for c in df.columns if c in ('station',)],
        'Đo lường'  : [c for c in df.columns if c.endswith('_nt')],
        'Chất lượng': [c for c in df.columns if 'quality' in c.lower() or 'flag' in c.lower()],
    }
    known = sum([col_groups[k] for k in col_groups], [])
    col_groups['Metadata'] = [c for c in df.columns if c not in known]

    for group, cols in col_groups.items():
        print(f'   {group:12s}: {cols}')


## 3. USGS – Nạp và khảo sát dữ liệu CSV động đất


In [ ]:
# Nạp dữ liệu thô danh mục động đất USGS từ file CSV
try:
    df_usgs_csv = load_usgs_csv()
    print(f'✅ USGS CSV: Nạp thành công {len(df_usgs_csv):,} sự kiện động đất')
    print(f'   Từ: {df_usgs_csv["time_utc"].min()}')
    print(f'   Đến: {df_usgs_csv["time_utc"].max()}')
    print(f'\n📌 Kiểu dữ liệu (Data Types):')
    print(df_usgs_csv.dtypes.to_string())
    print(f'\n📌 Thống kê mô tả:')
    print(df_usgs_csv.describe(include='all').T.to_string())
    print(f'\n📌 5 dòng dữ liệu đầu tiên:')
    print(df_usgs_csv.head().to_string())
except FileNotFoundError as e:
    print(f'⚠  {e}')
    df_usgs_csv = None


## 4. USGS – Nạp và khảo sát dữ liệu GeoJSON động đất


In [ ]:
# Nạp dữ liệu thô danh mục động đất USGS từ file GeoJSON và so sánh với CSV
try:
    df_usgs_gj = load_usgs_geojson()
    print(f'✅ USGS GeoJSON: Nạp thành công {len(df_usgs_gj):,} sự kiện động đất')
    print(f'   Từ: {df_usgs_gj["time_utc"].min()}')
    print(f'   Đến: {df_usgs_gj["time_utc"].max()}')
    print(f'\n📌 5 dòng dữ liệu đầu tiên:')
    print(df_usgs_gj.head().to_string())

    # So sánh số lượng sự kiện giữa CSV và GeoJSON
    if df_usgs_csv is not None:
        print(f'\n📌 So sánh CSV ({len(df_usgs_csv):,}) vs GeoJSON ({len(df_usgs_gj):,}):')
        print(f'   Chênh lệch: {len(df_usgs_csv) - len(df_usgs_gj)} sự kiện')
except FileNotFoundError as e:
    print(f'⚠  {e}')
    df_usgs_gj = None


## 5. Tóm tắt Bước 1 (Khảo sát cấu trúc KAK & USGS)

Tổng hợp lại kết quả khảo sát cấu trúc của 2 nguồn dữ liệu KAK và USGS.


In [ ]:
# ── Báo cáo tổng hợp Bước 1 ──────────────────────────────────────────────────
print('=' * 65)
print('  TÓM TẮT BƯỚC 1 – KHẢO SÁT CẤU TRÚC DỮ LIỆU THÔ')
print('=' * 65)

for station in cfg.STATIONS:
    if station not in station_dfs:
        continue
    df   = station_dfs[station]
    meta = station_meta2.get(station, {})
    meas = [c for c in df.columns if c.endswith('_nt')]

    print(f'\n[DỮ LIỆU ĐỊA TỪ KAK / {station}]')
    print(f'  Mã trạm     : {meta.get("station_code")}')
    print(f'  Tọa độ      : {meta.get("geodetic_lat")}°N, {meta.get("geodetic_lon")}°E')
    print(f'  Độ cao      : {meta.get("elevation")} m')
    print(f'  Thành phần  : {meta.get("reported_components")}')
    print(f'  Tần số mẫu  : {meta.get("sample_period_seconds")} giây')
    print(f'  Loại dữ liệu: {meta.get("data_type")}')
    print(f'  Giá trị Lỗi : {meta.get("sentinel_values")} (Sentinel theo chuẩn IAGA-2002)')
    print(f'  Tổng số dòng: {len(df):,}')
    print(f'  Khoảng TG   : {df["time_utc"].min()} → {df["time_utc"].max()}')
    print(f'  Cột đo lường: {meas}')

if df_usgs_csv is not None:
    print(f'\n[DỮ LIỆU ĐỘNG ĐẤT USGS CSV]')
    print(f'  Số sự kiện  : {len(df_usgs_csv):,}')
    print(f'  Khoảng TG   : {df_usgs_csv["time_utc"].min()} → {df_usgs_csv["time_utc"].max()}')
    print(f'  Độ lớn (Mag): {df_usgs_csv["magnitude"].min():.1f} – {df_usgs_csv["magnitude"].max():.1f}')
    print(f'  Danh sách cột: {list(df_usgs_csv.columns)}')

print('\n' + '─'*65)
print('→ Hoàn thành Bước 1. Tiếp tục chuyển sang Bước 2 (Kiểm tra chất lượng).')
